# 5장. 미니배치와 옵티마이저

가중치를 고치는 규칙을 바꾸면 무엇이 달라지고, 무엇은 그대로인가.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 두 개(2023년, 2025년)와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023년, 2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 4분쯤 걸린다.

## 5.1 배치 크기에 따른 기울기의 변동 재기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
import time
import torch
from torch import nn

def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

def column(values):
    """넘파이 배열을 (자료의 수, 1) 모양의 32비트 텐서로 바꾼다."""
    return torch.tensor(np.asarray(values)[:, None], dtype=torch.float32)

def pad(text, width):
    """글자 수를 맞춰 오른쪽을 공백으로 채운다. 한글은 두 칸으로 센다."""
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

torch.set_num_threads(1)
t_mean, t_std = T23.mean(), T23.std()
y_mean, y_std = Y23.mean(), Y23.std()
X, Y = column((T23 - t_mean) / t_std), column((Y23 - y_mean) / y_std)
X25 = column((T25 - t_mean) / t_std)
loss_fn = nn.MSELoss()

def make(seed):
    """은닉층 1개, 유닛 8개인 신경망. 시작값은 파이토치의 기본값이다."""
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))

def score(model):
    """2025년 RMSE(GW). 예측에 nan이 있으면 nan을 돌려준다."""
    with torch.no_grad():
        out = model(X25).numpy()[:, 0].astype(np.float64) * y_std + y_mean
    return rmse(Y25, out) if np.isfinite(out).all() else float("nan")

print(X.shape, Y.shape, sum(param.numel() for param in make(0).parameters()))

In [ ]:
def fit(model, optimizer, epochs, batch_size=32, seed=0, scheduler=None):
    """미니배치 학습. 에폭마다 247일 전체의 손실을 기록한다."""
    shuffler = torch.Generator().manual_seed(seed)
    trace = []
    for epoch in range(epochs):
        order = torch.randperm(len(X), generator=shuffler)     # 0~246을 섞은 순서
        for start in range(0, len(X), batch_size):
            rows = order[start:start + batch_size]
            optimizer.zero_grad()
            loss_fn(model(X[rows]), Y[rows]).backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()                                   # 5.4절에서 쓴다
        with torch.no_grad():
            trace.append(loss_fn(model(X), Y).item())
    return np.array(trace), score(model)

def trial(make_optimizer, epochs=200, batch_size=32, make_scheduler=None):
    """시드 0~4로 학습시킨다. (에폭마다의 손실 기록 5개, 2025년 RMSE 5개)를 돌려준다."""
    traces, marks = [], []
    for seed in range(5):
        model = make(seed)
        optimizer = make_optimizer(model.parameters())
        scheduler = make_scheduler(optimizer) if make_scheduler else None
        trace, mark = fit(model, optimizer, epochs, batch_size, seed, scheduler)
        traces.append(trace)
        marks.append(mark)
    return traces, np.array(marks)

def show(name, traces, marks):
    """마지막 손실의 범위, 2025년 RMSE의 중앙값과 범위, 발산한 시드의 수를 출력한다."""
    last = np.array([trace[-1] for trace in traces])
    good = np.isfinite(marks) & (last < 100)
    if good.any():
        last, ok = last[good], marks[good]
        text = (f"{last.min():.4f}~{last.max():.4f}   "
                f"{np.median(ok):.2f} ({ok.min():.2f}~{ok.max():.2f})")
    else:
        text = "-"
    print(pad(name, 30) + text + f"   발산 {5 - good.sum()}")

def sgd(rate, momentum=0.0):
    return lambda params: torch.optim.SGD(params, lr=rate, momentum=momentum)

show("배치 32, 학습률 0.05, 200에폭", *trial(sgd(0.05)))

In [ ]:
def flat_grad(model, rows):
    """고른 날들로 구한 기울기를 한 줄로 편 텐서."""
    model.zero_grad()
    loss_fn(model(X[rows]), Y[rows]).backward()
    return torch.cat([param.grad.ravel() for param in model.parameters()]).clone()

model = make(0)
whole = flat_grad(model, torch.arange(247))
picker = torch.Generator().manual_seed(0)
noise = {}
for size in [1, 2, 4, 8, 16, 32, 64, 128, 247]:
    gaps = []
    for _ in range(500):
        rows = torch.randperm(247, generator=picker)[:size]
        gap = (flat_grad(model, rows) - whole).norm() / whole.norm()
        gaps.append(gap.item() ** 2)
    noise[size] = float(np.sqrt(np.mean(gaps)))
print(" 크기   잰 값   식의 값")
for size, value in noise.items():
    formula = noise[1] * np.sqrt((247 - size) / (size * 246))
    print(f"{size:>5} {value:>7.3f} {formula:>8.3f}")

In [ ]:
print("같은 학습률 0.05, 같은 에폭 수 200")
same_epochs = {}
for size in [8, 32, 247]:
    began = time.perf_counter()
    same_epochs[size] = trial(sgd(0.05), 200, size)
    took = time.perf_counter() - began
    steps = 200 * int(np.ceil(247 / size))
    show(f"배치 {size}: {steps}걸음", *same_epochs[size])
    print(f"      {took:.1f}초")

In [ ]:
print("같은 학습률 0.05, 같은 걸음 수(약 1,600)")
for size, epochs in [(8, 52), (32, 200), (247, 1600)]:
    show(f"배치 {size}: {epochs}에폭", *trial(sgd(0.05), epochs, size))
print("배치 크기에 비례하는 학습률, 같은 에폭 수 200")
for size in [8, 32, 247]:
    rate = 0.05 * size / 32
    show(f"배치 {size}: 학습률 {rate:.4g}", *trial(sgd(rate), 200, size))

## 5.2 모멘텀으로 지난 기울기 이어 쓰기

In [ ]:
def velocities(grads, beta):
    """기울기가 차례로 들어올 때 속도 v가 어떻게 변하는지 계산한다."""
    v, out = 0.0, []
    for g in grads:
        v = beta * v + g
        out.append(round(v, 3))
    return out

print(velocities([1.0] * 5, 0.9), velocities([1.0] * 100, 0.9)[-1])
print(velocities([1.0, -1.0] * 3, 0.9), velocities([1.0, -1.0] * 50, 0.9)[-1])
print(velocities([1.0] * 5, 1.0))

In [ ]:
design = np.column_stack([T23, np.ones(len(T23))])           # 표준화하지 않은 기온과 1
solution = np.linalg.lstsq(design, Y23, rcond=None)[0]
floor = ((design @ solution - Y23) ** 2).mean()               # 가장 낮은 손실
curvature = np.linalg.eigvalsh(2 * design.T @ design / len(T23))

def line_loss(theta):
    return ((design @ theta - Y23) ** 2).mean()

def line_grad(theta):
    return 2 * design.T @ (design @ theta - Y23) / len(T23)

def descend(rate, beta=0.0, steps=5000):
    """직선 모델 w·x + c를 모멘텀으로 학습시킨다. beta = 0이면 경사하강법이다."""
    theta, v, trace = np.zeros(2), np.zeros(2), []
    for _ in range(steps):
        v = beta * v + line_grad(theta)
        theta = theta - rate * v
        trace.append(line_loss(theta))
        if not np.isfinite(trace[-1]) or trace[-1] > 1e12:
            break                                             # 발산
    return np.array(trace)

def reached(trace):
    """손실이 가장 낮은 손실의 1% 안으로 처음 들어온 걸음. 들어오지 못했으면 None."""
    close = trace < floor * 1.01
    return int(np.argmax(close)) + 1 if close.any() else None

def settled(trace):
    """그 뒤로 끝까지 1% 안에 머문 첫 걸음. 마지막 걸음에도 밖에 있으면 None."""
    outside = np.where(~(trace < floor * 1.01))[0]
    if len(outside) == 0:
        return 1
    return int(outside.max()) + 2 if outside.max() < len(trace) - 1 else None

print(solution.round(3).tolist(), round(floor, 2), curvature.round(3).tolist())
print(round(2 / curvature[1], 5), round(2 * 1.9 / curvature[1], 5))

In [ ]:
line_runs = {}
settings = [(0.003, 0.0), (0.0003, 0.9), (0.003, 0.9), (0.006, 0.9), (0.0061, 0.9)]
for rate, beta in settings:
    trace = descend(rate, beta)
    line_runs[rate, beta] = trace
    end = "발산" if len(trace) < 5000 else f"{trace[-1]:.2f}"
    first, stay = str(reached(trace)), str(settled(trace))
    print(f"학습률 {rate:<7g} 모멘텀 {beta:<4g} {first:>6} {stay:>6} {end:>8}")
plain, scaled = line_runs[0.003, 0.0] - floor, line_runs[0.0003, 0.9] - floor
ratio = scaled / plain                            # 두 설정의 (손실 − 가장 낮은 손실)의 비
print(round(ratio[:30].min(), 2), round(ratio[:30].max(), 2),
      round(ratio[30:2488].min(), 2), round(ratio[30:2488].max(), 2))
fast = line_runs[0.006, 0.9]
print(f"{fast.max():.3g}", int(fast.argmax()) + 1)

In [ ]:
print("모멘텀 없음")
for rate in [0.05, 0.01]:
    show(f"학습률 {rate}", *trial(sgd(rate)))
print("모멘텀 0.9")
for rate in [0.05, 0.005, 0.001]:
    show(f"학습률 {rate}", *trial(sgd(rate, 0.9)))
print("모멘텀 0.99")
show("학습률 0.05", *trial(sgd(0.05, 0.99)))
show("학습률 0.0005", *trial(sgd(0.0005, 0.99)))

## 5.3 Adam으로 가중치마다 걸음의 크기 맞추기

In [ ]:
def adam_steps(grads, rate=0.001, b1=0.9, b2=0.999, eps=1e-8):
    """가중치 하나에 기울기가 차례로 들어올 때 Adam이 걸음마다 옮기는 크기."""
    m, v, out = 0.0, 0.0, []
    for k, g in enumerate(grads, start=1):
        m = b1 * m + (1 - b1) * g                 # 기울기의 이동 평균
        v = b2 * v + (1 - b2) * g * g             # 기울기 제곱의 이동 평균
        m_hat, v_hat = m / (1 - b1 ** k), v / (1 - b2 ** k)
        out.append(rate * m_hat / (np.sqrt(v_hat) + eps))
    return [round(float(step), 6) for step in out]

print(adam_steps([0.1] * 4))
print(adam_steps([-20.0] * 4))
print(adam_steps([1e-9] * 4))
print(adam_steps([1.0, -1.0, 1.0, -1.0]))

In [ ]:
import copy

class MyAdam:
    """Adam을 식 그대로 적은 옵티마이저. torch.optim.Adam과 같은 방법으로 쓴다."""

    def __init__(self, params, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr, self.b1, self.b2, self.eps, self.k = lr, b1, b2, eps, 0
        self.m = [torch.zeros_like(param) for param in self.params]
        self.v = [torch.zeros_like(param) for param in self.params]

    def zero_grad(self):
        for param in self.params:
            param.grad = None

    def step(self):
        self.k += 1
        with torch.no_grad():
            for param, m, v in zip(self.params, self.m, self.v):
                m.mul_(self.b1).add_((1 - self.b1) * param.grad)
                v.mul_(self.b2).add_((1 - self.b2) * param.grad ** 2)
                m_hat = m / (1 - self.b1 ** self.k)
                v_hat = v / (1 - self.b2 ** self.k)
                param -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

ours, theirs = make(0), make(0)
trace_ours, _ = fit(ours, MyAdam(ours.parameters(), lr=0.01), 100)
trace_theirs, _ = fit(theirs, torch.optim.Adam(theirs.parameters(), lr=0.01), 100)
pairs = zip(ours.parameters(), theirs.parameters())
print(round(trace_ours[-1], 4), round(trace_theirs[-1], 4))
print(f"{max((a - b).abs().max().item() for a, b in pairs):.1e}")

In [ ]:
def adam_line(rate, steps=5000, b1=0.9, b2=0.999, eps=1e-8):
    """직선 모델을 Adam으로 학습시킨다. (손실 기록, 마지막 w와 c)를 돌려준다."""
    theta, m, v, trace = np.zeros(2), np.zeros(2), np.zeros(2), []
    for k in range(1, steps + 1):
        g = line_grad(theta)
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        theta = theta - rate * (m / (1 - b1 ** k)) / (np.sqrt(v / (1 - b2 ** k)) + eps)
        trace.append(line_loss(theta))
    return np.array(trace), theta

adam_runs = {}
for rate in [0.001, 0.01, 0.1, 1.0]:
    trace, theta = adam_line(rate)
    adam_runs[rate] = trace
    where = theta.round(2).tolist()
    first, stay = str(reached(trace)), str(settled(trace))
    print(f"학습률 {rate:<6g} {first:>6} {stay:>6} {trace[-1]:>9.2f}   {where}")
late = adam_runs[1.0][1000:]                      # 학습률 1에서 1,000걸음 뒤의 손실
print(round(late.min(), 2), round(late.max(), 2), int(late.argmax()) + 1001)

In [ ]:
X_raw, Y_raw, X25_raw = column(T23), column(Y23), column(T25)

def fit_raw(make_optimizer, seed, steps=3000):
    """표준화하지 않은 기온과 전력수요로, 전체 배치 학습을 한다(4.1절과 같은 조건)."""
    model = make(seed)
    optimizer = make_optimizer(model.parameters())
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X_raw), Y_raw)
        loss.backward()
        optimizer.step()
        if not torch.isfinite(loss):
            return float("nan")
    with torch.no_grad():
        return rmse(Y25, model(X25_raw).numpy()[:, 0].astype(np.float64))

def adam(rate):
    return lambda params: torch.optim.Adam(params, lr=rate)

marks = [fit_raw(sgd(0.0001), seed) for seed in range(5)]
print(pad("경사하강법, 학습률 0.0001", 28), [f"{mark:.2f}" for mark in marks])
for rate in [0.001, 0.01, 0.1, 1.0]:
    marks = [fit_raw(adam(rate), seed) for seed in range(5)]
    print(pad(f"Adam, 학습률 {rate}", 28), [f"{mark:.2f}" for mark in marks])

In [ ]:
def build(depth, seed, width=32):
    """4장의 build와 같다. 은닉층이 depth개이고 시작값은 파이토치의 기본값이다."""
    torch.manual_seed(seed)
    layers, m = [], 1
    for size in [width] * depth + [1]:
        layers += [nn.Linear(m, size), nn.ReLU()]
        m = size
    return nn.Sequential(*layers[:-1])

def train_deep(make_optimizer, seed, steps=2000):
    """4장의 train과 같은 조건: 20층, 전체 배치, 2,000걸음."""
    model = build(20, seed)
    optimizer = make_optimizer(model.parameters())
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        optimizer.step()
    return loss.item(), score(model)

deep = [("경사하강법, 학습률 0.02", sgd(0.02)), ("Adam, 학습률 0.0001", adam(0.0001)),
        ("Adam, 학습률 0.001", adam(0.001))]
for name, make_optimizer in deep:
    runs = [train_deep(make_optimizer, seed) for seed in range(5)]
    print(pad(name, 26) + " ".join(f"{loss:.4f}" for loss, mark in runs) + " | "
          + " ".join(f"{mark:.2f}" for loss, mark in runs))

## 5.4 학습하는 동안 학습률 바꾸기

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, MultiStepLR

def cosine(optimizer):
    return CosineAnnealingLR(optimizer, T_max=200)       # 200에폭에 걸쳐 0까지 줄인다

def stairs(optimizer):
    # 100에폭과 150에폭에서 학습률을 10분의 1로 줄인다
    return MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

def summary(name, traces, marks):
    """손실이 0.23 아래로 내려간 에폭(중앙값), 마지막 20에폭의 손실 폭, RMSE."""
    first = [int(np.argmax(trace < 0.23)) + 1 if trace.min() < 0.23 else 999
             for trace in traces]
    width = max(trace[-20:].max() - trace[-20:].min() for trace in traces)
    print(pad(name, 26) + f"{int(np.median(first)):>5} {width:>9.4f}   "
          f"{np.median(marks):.2f} ({marks.min():.2f}~{marks.max():.2f})")

plans = {"0.01로 고정": (0.01, None), "0.05로 고정": (0.05, None),
         "0.2로 고정": (0.2, None), "0.2에서 코사인": (0.2, cosine),
         "0.2에서 계단": (0.2, stairs)}
sched_runs = {}
for name, (rate, plan) in plans.items():
    sched_runs[name] = trial(sgd(rate), make_scheduler=plan)
    summary(name, *sched_runs[name])

In [ ]:
model = make(0)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
plan = cosine(optimizer)
rates = []
for epoch in range(200):
    rates.append(optimizer.param_groups[0]["lr"])              # 이번 에폭에 쓰는 학습률
    optimizer.step()
    plan.step()
print([round(rates[k], 4) for k in (0, 50, 100, 150, 199)])
print(round(0.2 * (1 + np.cos(np.pi * 50 / 200)) / 2, 4))

## 5.5 옵티마이저와 학습률 고르기

In [ ]:
rates = [0.0001, 0.001, 0.01, 0.1, 1.0]
kinds = {"경사하강법": lambda rate: sgd(rate),
         "모멘텀 0.9": lambda rate: sgd(rate, 0.9), "Adam": lambda rate: adam(rate)}
sweep = {}
print(pad("학습률", 14) + "".join(f"{rate:>13}" for rate in rates))
for name, kind in kinds.items():
    row = pad(name, 14)
    for rate in rates:
        traces, marks = trial(kind(rate))
        good = np.isfinite(marks) & (np.array([trace[-1] for trace in traces]) < 100)
        sweep[name, rate] = marks[good]
        middle = f"{np.median(marks[good]):.2f}" if good.any() else "-"
        row += f"{middle:>9} ({5 - good.sum()})"
    print(row)

In [ ]:
tuned = {"경사하강법, 0.2에서 코사인": sgd(0.2),
         "모멘텀 0.9, 0.02에서 코사인": sgd(0.02, 0.9),
         "Adam, 0.1에서 코사인": adam(0.1), "Adam, 0.001로 고정(기본값)": adam(0.001)}
best_runs = {}
for name, make_optimizer in tuned.items():
    plan = None if "고정" in name else cosine
    best_runs[name] = trial(make_optimizer, make_scheduler=plan)
    show(name, *best_runs[name])

In [ ]:
import sys, platform

print(sys.version.split()[0], np.__version__, torch.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 5-1
sizes = np.array(list(noise))
values = np.array(list(noise.values()))
smooth = np.linspace(1, 246, 400)
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.plot(smooth, noise[1] / np.sqrt(smooth), color=GRAY, lw=1.3, ls=(0, (5, 2)), label="배치 크기의 제곱근에 반비례")
ax.plot(smooth, noise[1] * np.sqrt((247 - smooth) / (smooth * 246)), color=NAVY, lw=1.6, label="식의 값 (전체가 247일인 것을 계산에 넣음)")
ax.scatter(sizes[:-1], values[:-1], s=42, color=BRICK, zorder=5, edgecolor="white", lw=0.8, label="잰 값 (배치를 500번 뽑아서 구함)")
for size in (1, 8, 32, 128):
    ax.text(size * 1.12, noise[size] * 1.12, f"{noise[size]:.2f}", fontsize=9, color=BRICK, ha="left", va="bottom")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(0.8, 300); ax.set_ylim(0.03, 6)
ax.set_xticks([1, 2, 4, 8, 16, 32, 64, 128, 247]); ax.set_xticklabels([1, 2, 4, 8, 16, 32, 64, 128, 247])
ax.set_yticks([0.05, 0.1, 0.2, 0.5, 1, 2, 5]); ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:g}"))
ax.minorticks_off()
ax.set_xlabel("배치 크기 (로그 축)", fontsize=9)
ax.set_ylabel("전체 기울기와의 차이 ÷ 전체 기울기의 크기 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="lower left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_1.png", dpi=300)
plt.show()

In [ ]:
# 그림 5-3
looks = {8: (BRICK, "-", 1.4), 32: (NAVY, "-", 1.8), 247: (TEAL, "-", 2.4)}
fig, axes = plt.subplots(2, 1, figsize=(7, 7.8))
for size, (col, ls, lw) in looks.items():
    trace = same_epochs[size][0][0]                      # 시드 0
    per_epoch = int(np.ceil(247 / size))
    name = f"배치 {size} (한 에폭에 {per_epoch}걸음)"
    axes[0].plot(np.arange(1, 201), trace, color=col, ls=ls, lw=lw, label=name)
    axes[1].plot(np.arange(1, 201) * per_epoch, trace, color=col, ls=ls, lw=lw, label=name)
for ax, label, hi in [(axes[0], "에폭 (자료 전체를 본 횟수, 로그 축)", 200), (axes[1], "가중치를 고친 횟수 (로그 축)", 6200)]:
    title = "① 자료 전체를 본 횟수로 본 손실" if hi == 200 else "② 가중치를 고친 횟수로 본 손실"
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(1, hi); ax.set_ylim(0.2, 1.2)
    plain_ticks(ax.xaxis); ax.set_yticks([0.2, 0.3, 0.5, 1])
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:g}")); ax.minorticks_off()
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("247일 전체의 손실 (로그 축)", fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    frame(ax, title)
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_5_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 5-5
looks = {(0.003, 0.0): (GRAY, "-", 2.0, "경사하강법, 학습률 0.003"), (0.0003, 0.9): (NAVY, (0, (5, 2)), 1.5, "모멘텀 0.9, 학습률 0.0003"),
         (0.003, 0.9): (TEAL, "-", 2.4, "모멘텀 0.9, 학습률 0.003"), (0.006, 0.9): (BRICK, "-", 1.5, "모멘텀 0.9, 학습률 0.006")}
fig, ax = plt.subplots(figsize=(7, 4.8))
for key, (col, ls, lw, name) in looks.items():
    trace = line_runs[key]
    ax.plot(np.arange(1, len(trace) + 1), np.maximum(trace - floor, 1e-9), color=col, ls=ls, lw=lw, label=f"{name}: {reached(trace):,} / {settled(trace):,}걸음")
ax.axhline(floor * 0.01, color=GRAY, lw=0.9, ls=(0, (2, 2)))
ax.text(1.2, floor * 0.01 * 1.25, "가장 낮은 손실의 1%", fontsize=9, color="#5b6470", ha="left", va="bottom")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1, 5000); ax.set_ylim(1e-6, 1e7)
plain_ticks(ax.xaxis); ax.set_yticks([1e-6, 1e-4, 1e-2, 1, 1e2, 1e4, 1e6])
ax.set_yticklabels(["0.000001", "0.0001", "0.01", "1", "100", "10,000", "1,000,000"]); ax.minorticks_off()
ax.set_xlabel("가중치를 고친 횟수 (로그 축)", fontsize=9)
ax.set_ylabel("손실 − 가장 낮은 손실 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right", title="1% 안에 처음 들어온 걸음 / 그 뒤로 머문 걸음", title_fontsize=9)
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_5.png", dpi=300)
plt.show()

In [ ]:
# 그림 5-7
looks = {0.001: (BRICK, (0, (1.5, 1.5)), 1.5), 0.01: (BRICK, (0, (5, 2)), 1.5), 0.1: (NAVY, "-", 1.7), 1.0: (TEAL, "-", 2.4)}
fig, ax = plt.subplots(figsize=(7, 4.8))
base = line_runs[0.003, 0.0]
ax.plot(np.arange(1, 5001), np.maximum(base - floor, 1e-9), color=GRAY, lw=2.0, label="경사하강법, 학습률 0.003")
for rate, (col, ls, lw) in looks.items():
    trace = adam_runs[rate]
    tail = f"{reached(trace):,} / {settled(trace):,}걸음" if reached(trace) else f"5,000걸음 뒤 손실 {trace[-1]:,.0f}"
    ax.plot(np.arange(1, 5001), np.maximum(trace - floor, 1e-9), color=col, ls=ls, lw=lw, label=f"Adam, 학습률 {rate:g}: {tail}")
ax.axhline(floor * 0.01, color=GRAY, lw=0.9, ls=(0, (2, 2)))
ax.text(1.2, floor * 0.01 * 1.25, "가장 낮은 손실의 1%", fontsize=9, color="#5b6470", ha="left", va="bottom")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1, 5000); ax.set_ylim(1e-6, 1e7)
plain_ticks(ax.xaxis); ax.set_yticks([1e-6, 1e-4, 1e-2, 1, 1e2, 1e4, 1e6])
ax.set_yticklabels(["0.000001", "0.0001", "0.01", "1", "100", "10,000", "1,000,000"]); ax.minorticks_off()
ax.set_xlabel("가중치를 고친 횟수 (로그 축)", fontsize=9)
ax.set_ylabel("손실 − 가장 낮은 손실 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="lower left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 5-8
epochs = np.arange(1, 201)
shapes = {"0.01로 고정": np.full(200, 0.01), "0.2로 고정": np.full(200, 0.2),
          "0.2에서 코사인": 0.2 * (1 + np.cos(np.pi * (epochs - 1) / 200)) / 2,
          "0.2에서 계단": np.where(epochs <= 100, 0.2, np.where(epochs <= 150, 0.02, 0.002))}
looks = {"0.01로 고정": (GRAY, "-", 1.6), "0.2로 고정": (BRICK, "-", 1.3), "0.2에서 코사인": (TEAL, "-", 2.4), "0.2에서 계단": (NAVY, (0, (5, 2)), 1.6)}
fig, axes = plt.subplots(2, 1, figsize=(7, 7.8))
for name, (col, ls, lw) in looks.items():
    axes[0].plot(epochs, shapes[name], color=col, ls=ls, lw=lw, label=name)
    axes[1].plot(epochs, sched_runs[name][0][0], color=col, ls=ls, lw=lw, label=name)
axes[0].set_ylim(0, 0.22); axes[0].set_ylabel("학습률", fontsize=9)
axes[1].set_yscale("log"); axes[1].set_ylim(0.2, 1.2); axes[1].set_yticks([0.2, 0.3, 0.5, 1])
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:g}")); axes[1].minorticks_off()
axes[1].set_ylabel("247일 전체의 손실 (로그 축, 시드 0)", fontsize=9)
for ax in axes:
    ax.set_xlim(1, 200); ax.set_xticks([1, 25, 50, 75, 100, 125, 150, 175, 200])
    ax.set_xlabel("에폭", fontsize=9)
    spot = dict(loc="center left", bbox_to_anchor=(0.03, 0.36)) if ax is axes[0] else dict(loc="upper right")
    ax.legend(frameon=False, fontsize=9, ncol=2, **spot)
    frame(ax, "① 에폭마다 쓴 학습률" if ax is axes[0] else "② 그 학습률로 학습한 손실")
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_5_8.png", dpi=300)
plt.show()

In [ ]:
# 그림 5-9
looks = {"경사하강법": (NAVY, "o", -0.035), "모멘텀 0.9": (BRICK, "s", 0.0), "Adam": (TEAL, "D", 0.035)}
fig, ax = plt.subplots(figsize=(7, 4.8))
for name, (col, marker, shift) in looks.items():
    xs_, mids = [], []
    for rate in rates:
        marks = sweep[name, rate]
        x = 10 ** (np.log10(rate) + shift)
        if len(marks) == 0:
            ax.text(10 ** (np.log10(rate) + shift * 5 - 0.12), 8.3, f"{name}: 발산", fontsize=9, color=col, ha="center", va="top", rotation=90)
            continue
        ax.plot([x, x], [marks.min(), min(marks.max(), 7.4)], color=col, lw=1.2)
        xs_.append(x); mids.append(np.median(marks))
    best = int(np.argmin(mids))
    ax.plot(xs_, mids, color=col, lw=1.6, marker=marker, ms=5.5, label=f"{name}: 가장 낮은 중앙값 {mids[best]:.2f}GW")
    ax.scatter([xs_[best]], [mids[best]], s=120, facecolor="none", edgecolor=col, lw=1.2, zorder=6)
ax.axhline(6.58, color=GRAY, lw=0.9, ls=(0, (2, 2)))
ax.text(0.02, 6.63, "평균으로 예측: 6.58GW", fontsize=9, color="#5b6470", ha="center", va="bottom")
ax.set_xscale("log")
ax.set_xlim(0.00006, 1.6); ax.set_ylim(2.8, 8.4)
ax.set_xticks(rates); ax.set_xticklabels([f"{rate:g}" for rate in rates]); ax.minorticks_off()
ax.set_xlabel("학습률 (로그 축)", fontsize=9)
ax.set_ylabel("2025년 RMSE (GW). 점은 시드 5개의 중앙값, 세로선은 범위", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, 1.0))
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_9.png", dpi=300)
plt.show()